# Linear probe vs. kNN, all experts

Replaces `probe_one_generator.ipynb` (ADM only). For every expert (`resnet/experts.py`), on the saved CLIP ViT-L/14 features:

- **Train:** the expert's `expert_train` rows (`expert_rows`, the same rows as `clip/train.py`).
- **Evaluate:** `expert_val` of **every** generator. The expert's own generators are in-distribution, the others are transfer.
- **Logistic regression:** StandardScaler + LogisticRegression for several `C` (`C = 1` is what the CLIP experts use).
- **kNN:** cosine distance (like Ojha et al.), several `k`, uniform or distance-weighted votes. The score is the (weighted) share
  of fake neighbours. The neighbours are searched once per expert with the largest `k`, and every smaller `k` reuses them.

Predicted fake = score > 0.5, as in `probe_one_generator.ipynb`. Accuracy is the main metric; AUC is coarse for kNN with small `k`.
The progress log prints an ETA after every step, so the settings below can grow.

In [8]:
import sys
sys.path.insert(0, "../..")  # repo root, so `clip`, `common`, `data` and `resnet` are importable

import time

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from clip.extract import FEATURE_NAME
from common.log import Progress, format_duration, log
from data import config
from data.features import load_features
from data.images import load_manifest
from resnet.experts import EXPERTS, expert_rows

LR_CS = [0.01, 0.1, 1.0, 10.0]  # C = 1 is what the CLIP experts use (clip/train.py)
MAX_ITER = 2000
KNN_KS = [1, 3, 5, 9, 15, 25, 51, 101, 201, 501, 999]  # odd, so uniform votes never tie
KNN_WEIGHTS = ["uniform", "distance"]  # distance: votes weighted by 1 / cosine distance
EVAL_POOL = "expert_val"

## Data

Features are loaded once per pool, and each expert takes its rows from them.

In [9]:
manifest = load_manifest()
X_train_all, rows_train_all = load_features(FEATURE_NAME, pool="expert_train")
X_eval_all, rows_eval_all = load_features(FEATURE_NAME, pool=EVAL_POOL)

# generator -> its EVAL_POOL features and labels
X_eval = {g: X_eval_all[(rows_eval_all.generator == g).to_numpy()] for g in config.GENERATORS}
y_eval = {g: rows_eval_all.label.to_numpy()[(rows_eval_all.generator == g).to_numpy()] for g in config.GENERATORS}
print("train pool", X_train_all.shape, "| eval images per generator", {g: len(y) for g, y in y_eval.items()})

train pool (80000, 768) | eval images per generator {'ADM': 2000, 'BigGAN': 2000, 'glide': 2000, 'Midjourney': 2000, 'stable_diffusion_v_1_4': 2000, 'stable_diffusion_v_1_5': 2000, 'VQDM': 2000, 'wukong': 2000}


## Run

In [10]:
results = []


def record(expert, model, param, p_fake_by_generator):
    """One result row per evaluated generator."""
    for generator, p_fake in p_fake_by_generator.items():
        y = y_eval[generator]
        predicted = p_fake > 0.5
        results.append({
            "expert": expert, "model": model, "param": param, "generator": generator,
            "own": generator in EXPERTS[expert],
            "acc": float((predicted == (y == 1)).mean()),
            "auc": float(roc_auc_score(y, p_fake)),
            "real correct": float((~predicted[y == 0]).mean()),
            "fake correct": float(predicted[y == 1].mean()),
        })


# One step per logistic regression fit and one per kNN neighbour search.
progress = Progress(len(EXPERTS) * (len(LR_CS) + 1))


def step_done(what, start):
    progress.step()
    remaining = progress.remaining_seconds()
    finish = time.strftime("%H:%M", time.localtime(time.time() + remaining))
    log(f"[{what}] {time.time() - start:.1f}s | {progress.done}/{progress.total} steps, "
        f"ETA {format_duration(remaining)}, done at ~{finish}")


run_start = time.time()
for expert in EXPERTS:
    train_rows = expert_rows(manifest, expert, "expert_train")
    X_train = X_train_all[rows_train_all.index.get_indexer(train_rows.index)]
    y_train = train_rows.label.to_numpy()

    for C in LR_CS:
        start = time.time()
        model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=MAX_ITER)).fit(X_train, y_train)
        record(expert, "logistic regression", f"C={C:g}", {g: model.predict_proba(X)[:, 1] for g, X in X_eval.items()})
        iterations = int(model[-1].n_iter_[0])
        step_done(f"{expert} LR C={C:g}, {iterations} iterations{' (NOT converged)' if iterations >= MAX_ITER else ''}", start)

    start = time.time()
    neighbours = NearestNeighbors(metric="cosine").fit(X_train)
    found = {g: neighbours.kneighbors(X, n_neighbors=max(KNN_KS)) for g, X in X_eval.items()}  # (distances, indices)
    for weights in KNN_WEIGHTS:
        for k in KNN_KS:
            p_fake = {}
            for g, (distances, indices) in found.items():
                votes = y_train[indices[:, :k]]
                w = np.ones_like(distances[:, :k]) if weights == "uniform" else 1 / np.maximum(distances[:, :k], 1e-12)
                p_fake[g] = (w * votes).sum(axis=1) / w.sum(axis=1)
            record(expert, f"kNN {weights}", f"k={k}", p_fake)
    step_done(f"{expert} kNN, {len(KNN_WEIGHTS) * len(KNN_KS)} settings", start)

results = pd.DataFrame(results)
results["eval"] = np.where(results.own, "in-distribution", "transfer")
log(f"done: {len(results)} results in {format_duration(time.time() - run_start)}")

[21:11:15] [ADM LR C=0.01, 26 iterations] 0.2s | 1/35 steps, ETA 0h00m07s, done at ~21:11
[21:11:15] [ADM LR C=0.1, 38 iterations] 0.2s | 2/35 steps, ETA 0h00m07s, done at ~21:11
[21:11:15] [ADM LR C=1, 70 iterations] 0.4s | 3/35 steps, ETA 0h00m08s, done at ~21:11
[21:11:16] [ADM LR C=10, 116 iterations] 0.9s | 4/35 steps, ETA 0h00m13s, done at ~21:11
[21:11:21] [ADM kNN, 22 settings] 4.7s | 5/35 steps, ETA 0h00m38s, done at ~21:11
[21:11:21] [BigGAN LR C=0.01, 25 iterations] 0.2s | 6/35 steps, ETA 0h00m31s, done at ~21:11
[21:11:21] [BigGAN LR C=0.1, 29 iterations] 0.2s | 7/35 steps, ETA 0h00m27s, done at ~21:11
[21:11:21] [BigGAN LR C=1, 21 iterations] 0.2s | 8/35 steps, ETA 0h00m23s, done at ~21:11
[21:11:21] [BigGAN LR C=10, 16 iterations] 0.1s | 9/35 steps, ETA 0h00m20s, done at ~21:11
[21:11:26] [BigGAN kNN, 22 settings] 4.3s | 10/35 steps, ETA 0h00m28s, done at ~21:11
[21:11:26] [glide LR C=0.01, 23 iterations] 0.1s | 11/35 steps, ETA 0h00m25s, done at ~21:11
[21:11:26] [glide 

## Every setting, averaged over experts

Accuracy, mean over the expert's own generators (in-distribution) and over all other generators (transfer).

In [14]:
summary = results.groupby(["expert", "model", "param", "eval"]).acc.mean().unstack("eval")
summary.groupby(["model", "param"]).mean().sort_values("transfer", ascending=False).round(3)

eval                        in-distribution  transfer
model               param                            
logistic regression C=0.01            0.973     0.740
                    C=0.1             0.971     0.737
                    C=1               0.965     0.735
                    C=10              0.962     0.732
kNN distance        k=51              0.798     0.700
                    k=25              0.812     0.698
kNN uniform         k=51              0.793     0.698
                    k=25              0.807     0.696
kNN distance        k=101             0.780     0.696
                    k=15              0.814     0.694
kNN uniform         k=101             0.775     0.693
                    k=15              0.809     0.692
kNN distance        k=201             0.756     0.687
                    k=9               0.818     0.687
kNN uniform         k=201             0.751     0.685
                    k=9               0.814     0.685
kNN distance        k=5               0.819     0.679
kNN uniform         k=5               0.818     0.678
kNN distance        k=3               0.824     0.674
kNN uniform         k=3               0.824     0.673
kNN distance        k=501             0.723     0.670
kNN uniform         k=501             0.719     0.668
                    k=1               0.825     0.661
kNN distance        k=1               0.825     0.661
                    k=999             0.699     0.656
kNN uniform         k=999             0.693     0.653

## Headline: logistic regression (C = 1) vs. best kNN

For each expert and column, the **best** kNN setting is picked on the evaluation images themselves. This favours kNN,
so "logistic regression wins" is a conservative statement. Positive gap = logistic regression is better.

In [15]:
lr = summary.xs(("logistic regression", "C=1"), level=("model", "param"))
best_knn = summary[summary.index.get_level_values("model").str.startswith("kNN")].groupby("expert").max()
gap = (lr - best_knn).loc[list(EXPERTS)]

print(f"in-distribution: LR minus best kNN {100 * gap['in-distribution'].min():+.1f} to "
      f"{100 * gap['in-distribution'].max():+.1f} points (mean {100 * gap['in-distribution'].mean():+.1f})")
print(f"transfer:        LR minus best kNN {100 * gap['transfer'].min():+.1f} to "
      f"{100 * gap['transfer'].max():+.1f} points (mean {100 * gap['transfer'].mean():+.1f})")

pd.concat({"logistic regression C=1": lr.loc[list(EXPERTS)], "best kNN": best_knn.loc[list(EXPERTS)],
           "gap": gap}, axis=1).round(3)

in-distribution: LR minus best kNN +9.4 to +14.5 points (mean +11.8)
transfer:        LR minus best kNN -10.1 to +10.6 points (mean +2.5)


logistic regression C=1                 best kNN           \
eval               in-distribution transfer in-distribution transfer   
expert                                                                 
ADM                          0.959    0.781           0.862    0.729   
BigGAN                       0.994    0.597           0.886    0.698   
glide                        0.984    0.666           0.838    0.712   
Midjourney                   0.956    0.715           0.863    0.708   
SD                           0.962    0.764           0.819    0.691   
VQDM                         0.958    0.809           0.845    0.723   
wukong                       0.942    0.812           0.821    0.706   

                       gap           
eval       in-distribution transfer  
expert                               
ADM                  0.097    0.052  
BigGAN               0.109   -0.101  
glide                0.145   -0.046  
Midjourney           0.094    0.006  
SD                   0.143    0.073  
VQDM                 0.114    0.086  
wukong               0.121    0.106

## Appendix: accuracy per expert and generator

Logistic regression with C = 1, and the single kNN setting with the best mean transfer accuracy over all experts.

In [16]:
knn_transfer = results[results.model.str.startswith("kNN") & ~results.own].groupby(["model", "param"]).acc.mean()
best_model, best_param = knn_transfer.idxmax()

for model, param in [("logistic regression", "C=1"), (best_model, best_param)]:
    table = results[(results.model == model) & (results.param == param)].pivot(index="expert", columns="generator", values="acc")
    print(f"{model}, {param}")
    display(table.loc[list(EXPERTS), config.GENERATORS].round(3))

logistic regression, C=1


generator,ADM,BigGAN,glide,Midjourney,stable_diffusion_v_1_4,stable_diffusion_v_1_5,VQDM,wukong
expert,,,,,,,,
ADM,0.959,0.856,0.904,0.618,0.748,0.744,0.906,0.694
BigGAN,0.593,0.994,0.906,0.518,0.512,0.514,0.624,0.513
glide,0.712,0.946,0.984,0.549,0.598,0.601,0.683,0.572
Midjourney,0.579,0.600,0.674,0.956,0.879,0.886,0.600,0.784
SD,0.700,0.642,0.749,0.839,0.962,0.963,0.763,0.892
VQDM,0.880,0.908,0.866,0.668,0.792,0.788,0.958,0.758
wukong,0.692,0.730,0.764,0.809,0.944,0.944,0.802,0.942


kNN distance, k=51


generator,ADM,BigGAN,glide,Midjourney,stable_diffusion_v_1_4,stable_diffusion_v_1_5,VQDM,wukong
expert,,,,,,,,
ADM,0.852,0.784,0.798,0.520,0.650,0.663,0.804,0.614
BigGAN,0.763,0.812,0.800,0.676,0.611,0.628,0.786,0.623
glide,0.710,0.732,0.736,0.670,0.709,0.718,0.694,0.700
Midjourney,0.600,0.682,0.666,0.820,0.780,0.789,0.656,0.780
SD,0.652,0.636,0.692,0.736,0.760,0.754,0.664,0.728
VQDM,0.784,0.812,0.730,0.662,0.648,0.669,0.842,0.667
wukong,0.655,0.627,0.700,0.758,0.774,0.760,0.666,0.766
